# Infinity-2B GGUF Figure 10 quantitative-style sample

This notebook is now a thin runner around the `var_soict` Python package in `VAR_SOICT/src`.

It uses the 10 Figure 10 styles marked for quantitative evaluation and randomly samples 3 Parti prompts for each style:

```text
10 styles x 3 prompts = 30 cases per generation step
```

Flow:

```text
Infinity-2B baseline
PFB + SAC
Multi-step PFB + SAC with decay at steps 2, 4, 6, 8
Top-1 SVD PFB + SAC with style-related steps
Final aggregate comparison
Download outputs
```


## 1. Import project modules and build configuration

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents[:5], Path('/content/VAR_SOICT')]
VAR_SOICT_ROOT = next(
    candidate.resolve() for candidate in candidates
    if (candidate / 'src' / 'var_soict').exists()
)
sys.path.insert(0, str(VAR_SOICT_ROOT / 'src'))

from var_soict.config import ExperimentConfig
from var_soict.bootstrap import build_runtime_paths

config = ExperimentConfig(
    infinity_source_dir=VAR_SOICT_ROOT / 'Infinity',
    download_missing_model_files=True,
)
paths = build_runtime_paths(config)

print('VAR_SOICT root:', VAR_SOICT_ROOT)
print('Runtime root:', paths.runtime_root)
print('Output dir:', paths.output_dir)
print('Official Infinity source:', paths.official_dir)
print('GGUF runtime cache:', paths.root)
print('MODEL_PN:', config.model_pn, '| CFG:', config.cfg, '| TAU:', config.tau, '| SEED:', config.seed, '| T5:', config.t5_device)


## 2. Check runtime

In [ ]:
from var_soict.bootstrap import check_torch_runtime

DEVICE = check_torch_runtime(require_cuda=True)


## 3. Install dependencies

In [ ]:
from var_soict.bootstrap import install_dependencies

install_dependencies()


## 4. Download and patch Infinity-2B GGUF assets

In [ ]:
from var_soict.bootstrap import import_gguf_loader, prepare_infinity_sources, validate_infinity_runtime_imports, verify_model_files

model_files = prepare_infinity_sources(config, paths)
verify_model_files(paths, model_files)
gguf_loader = import_gguf_loader(paths, model_files)
validate_infinity_runtime_imports(paths)


## 5. Load model components and scale schedule

In [ ]:
from var_soict.bootstrap import build_scale_schedule, load_model_bundle

bundle = load_model_bundle(config, model_files, gguf_loader)
bundle.scale_schedule = build_scale_schedule(config.model_pn, aspect_ratio=1.0)


## 6. Sample prompts and create experiment runner

In [ ]:
from var_soict.dataset import load_random_eval_cases
from var_soict.experiment import Infinity2BExperiment
from var_soict.style_transfer import StyleTransferEngine

sessions, cases, selected_cases_csv = load_random_eval_cases(
    var_soict_root=VAR_SOICT_ROOT,
    output_dir=paths.output_dir,
    config=config,
)
engine = StyleTransferEngine(bundle, config)
experiment = Infinity2BExperiment(
    engine=engine,
    sessions=sessions,
    cases=cases,
    config=config,
    paths=paths,
)

print(f'Quantitative styles: {len(sessions)}')
print(f'Random prompts per style: {config.prompts_per_style}')
print(f'Cases per generation step: {len(cases)}')
print('Selected case manifest:', selected_cases_csv)
for session in sessions:
    print(f"- {session['style_id']} | Figure 10 #{session['figure10_index']:02d} | {session['style_label']} | {len(session['prompts'])} prompts")


## 7. Step 1: Baseline images from Infinity-2B

In [ ]:
baseline_results = experiment.run_baseline()


## 8. Step 2: PFB + SAC

In [ ]:
pfb_sac_results = experiment.run_pfb_sac()


## 9. Step 3: Multi-step PFB + SAC with decay at 2, 4, 6, 8

In [ ]:
multistep_results = experiment.run_multistep_decay()


## 10. Step 4: Top-1 SVD PFB + SAC with style-related steps

In [ ]:
top1_style_steps_results = experiment.run_top1_style_steps()


## Output layout

The notebook writes results under:

```text
Infinity_outputs/infinity2b_random_3_prompts_per_eval_style/
```

Each generation step saves 30 per-case PNGs: 10 quantitative-eval styles x 3 randomly sampled Parti prompts per style. The selected prompt/style pairs are saved to:

```text
selected_cases_30.csv
```

The final aggregate section produces one row per sampled prompt with this format:

```text
style reference | baseline | PFB + SAC | multi-step decay | top-1 style steps
```

The prompt text is shown below the style reference image for that row.


## 11. Final aggregate comparison

In [ ]:
experiment.plot_final_aggregate()


## Download outputs

Run this final cell in Colab to download the complete experiment output folder as a ZIP file.


In [ ]:
zip_path = experiment.download_outputs()
print('Downloaded ZIP:', zip_path)
